`Dataset` https://graph.openaire.eu/docs/

`API` https://api.openaire.eu/graph/swagger-ui/index.html#/Research%20products/search

`organizations` We focus on the *University of Pisa*, so we collect all organizations whose name includes "Pisa".

In [ ]:
import requests
import json
import os
import time
from typing import List, Dict

In [ ]:
# utils variables
url = "https://api.openaire.eu/graph/v1/organizations?search=Pisa&page=1&pageSize=100&sortBy=relevance%20DESC"
headers = {"accept": "application/json"}

_OUTPUT_DIR_ORGS_ = "./output/organizations"
os.makedirs(_OUTPUT_DIR_ORGS_, exist_ok=True)
_OUTPUT_DIR_PRODUCTS_ = "./output/products"
os.makedirs(_OUTPUT_DIR_PRODUCTS_, exist_ok=True)

In [ ]:
# Fetch organizations from OpenAIRE API
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
data_Pisa = response.json()

# Display response summary
total_records = len(data_Pisa.get('results', []))
print(f"[RESPONSE] Loaded {total_records} records. Keys: {data_Pisa.keys()}")

# Extract unique legalName values
unique_legal_names = sorted({
    entry.get("legalName") 
    for entry in data_Pisa["results"] 
    if entry.get("legalName")
})

# Print as Python array for copy-paste
print(f"\n[legalName unique] {len(unique_legal_names)}")
print("legal_names = [")
for name in unique_legal_names:
    print(f'    "{name}",')
print("]")

In [ ]:
legal_names = [
    "Azienda Ospedaliero Universitaria Pisana",
    "Department of Chemistry and Industrial Chemistry - Università di Pisa",
    "Department of Civilization and forms of knowledge - University of Pisa",
    "Dipartimento di Fisica Università degli Studi di Pisa",
    "Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa",
    "Dipartimento di Studi Italianistici - Università di Pisa",
    "Dipartimento di scienze economiche - Università di Pisa",
    "Research Center E. Piaggio - University of Pisa",
    "University of Pisa",
    "University of Pisa, Department of Clinical and Experimental Medecine, Division of Pharmacology, Pisa, Italy",
    "University of Toronto Università degli Studi di Pisa",
    "University of Toronto Università degli Studi di Pisa Universität Wien",
    "Università degli Studi di Pisa - Dipartimento die Matematica",
]

In [ ]:
# Filter organizations by legal_names and save to file
original_count = len(data_Pisa["results"])
filtered_data_Pisa = {
    **data_Pisa,
    "results": [e for e in data_Pisa["results"] if e.get("legalName") in legal_names]
}

filtered_count = len(filtered_data_Pisa["results"])
removed_count = original_count - filtered_count

# Save to file and display results
output_file = os.path.join(_OUTPUT_DIR_ORGS_, "pisa_organizations.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(filtered_data_Pisa['results'], f, ensure_ascii=False, indent=2)

print(f"[FILTERED] From {original_count} to {filtered_count} entries ([REMOVED] {removed_count})")
print(f"[SAVED] {output_file}")

In [ ]:
class ResearchProductsFetcher:
    def __init__(self, _OUTPUT_DIR_=None):
        if _OUTPUT_DIR_ is None:
            _OUTPUT_DIR_ = "./output/products"
        self.api_url = "https://api.openaire.eu/graph/v2/researchProducts"
        self._OUTPUT_DIR_ = _OUTPUT_DIR_
        self.headers = {"accept": "application/json"}
        os.makedirs(_OUTPUT_DIR_, exist_ok=True)

    def fetch_all(self, organizations: List[Dict]) -> Dict:
        products = {}
        for org in organizations:
            org_id = org.get("id")
            org_name = org.get("legalShortName") or org.get("legalName") or "Unknown"
            if not org_id:
                print(f"[ORG.] No ID: {org_name}")
                continue
            print(f"\nProcessing: {org_name} (ID: {org_id})")
            for prod in self._fetch_by_cursor(org_id):
                prod_id = prod.get("id")
                if not prod_id:
                    continue
                if prod_id not in products:
                    prod["affiliated_organizations"] = [{"id": org_id, "name": org_name}]
                    products[prod_id] = prod
                else:
                    products[prod_id].setdefault("affiliated_organizations", []).append({"id": org_id, "name": org_name})
            print(f"[FOUND] {len(products)} unique products so far")
            time.sleep(0.5)
        return products

    def _fetch_by_cursor(self, org_id: str) -> List[Dict]:
        all_products = []
        cursor = "*"
        while cursor:
            params = {
                "relOrganizationId": org_id,
                "cursor": cursor,
                "pageSize": 100,
                "sortBy": "publicationDate DESC"
            }
            try:
                resp = requests.get(self.api_url, params=params, headers=self.headers, timeout=30)
                resp.raise_for_status()
                data = resp.json()
                results = data.get("results", [])
                if not results:
                    break
                all_products.extend(results)
                cursor = data.get("header", {}).get("nextCursor")
                print(f"  [FETCHED] {len(all_products)} products")
                time.sleep(0.2 if len(all_products) < 5000 else 0.8)
            except Exception as e:
                print(f"  [Error] {e}")
                break
        return all_products

    def save(self, products: Dict, prefix="research_products"):
        products_list = list(products.values())
        main_file = os.path.join(self._OUTPUT_DIR_, f"{prefix}_complete.json")
        with open(main_file, "w", encoding="utf-8") as f:
            json.dump(products_list, f, indent=2, ensure_ascii=False)
        print(f"\n[SUCCESS] Saved {len(products_list)} products to: {main_file}")

        compact_file = os.path.join(self._OUTPUT_DIR_, f"{prefix}_compact.json")
        compact = [{
            "id": prod.get("id"), "mainTitle": prod.get("mainTitle"),
            "publicationDate": prod.get("publicationDate"), "type": prod.get("type"),
            "doi": next((pid.get("value") for pid in prod.get("pid", []) if pid.get("scheme") == "doi"), None),
            "affiliated_organizations": prod.get("affiliated_organizations", [])
        } for prod in products_list]
        with open(compact_file, "w", encoding="utf-8") as f:
            json.dump(compact, f, indent=2, ensure_ascii=False)
        print(f"[SUCCESS] Saved compact version to: {compact_file}")

        self._save_stats(products_list)

    def _save_stats(self, products: List[Dict]):
        stats = {
            "total": len(products),
            "by_type": {},
            "by_year": {},
            "with_doi": 0,
            "organizations": set()
        }
        for prod in products:
            t = prod.get("type", "unknown")
            stats["by_type"][t] = stats["by_type"].get(t, 0) + 1
            if prod.get("publicationDate"):
                y = prod["publicationDate"][:4]
                stats["by_year"][y] = stats["by_year"].get(y, 0) + 1
            if any(pid.get("scheme") == "doi" for pid in prod.get("pid", [])):
                stats["with_doi"] += 1
            for org in prod.get("affiliated_organizations", []):
                stats["organizations"].add(org["name"])
        stats["organizations"] = list(stats["organizations"])
        stats_file = os.path.join(self._OUTPUT_DIR_, "statistics.json")
        with open(stats_file, "w", encoding="utf-8") as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)
        print(f"[SUCCESS] Stats saved to: {stats_file}")
        print("\nSUMMARY:")
        print(f"     [Total] products: {stats['total']}")
        print(f"     [With DOI] products: {stats['with_doi']}")
        print(f"     [Organizations] count: {len(stats['organizations'])}")
        print(f"     [Types] distribution: {stats['by_type']}")

In [ ]:
fetcher = ResearchProductsFetcher(_OUTPUT_DIR_=_OUTPUT_DIR_PRODUCTS_)
with open(os.path.join(_OUTPUT_DIR_ORGS_, "pisa_organizations.json"), "r", encoding="utf-8") as f:
    fetcher.save(fetcher.fetch_all(json.load(f)))